# N016 · 问题建模与竞争解法反例

**目标：** 区分连续、非连续、计数和最优化等不同任务。

**先修：** N012, N013, N015。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 输入/输出契约；子数组与子序列；必要/充分条件；贪心反例。

**配套讲解来源：** [同名逐章意见](../../comment/016_problem_modeling_and_counterexamples.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

很多题做错，不是算法写错，而是从一开始就把"要枚举的东西"搞错了。题目让你处理"连续的一段"，你却去枚举"任意挑几个"；或者反过来。这章专门帮你把这两类对象区分清楚，并教你亲手把两类东西全部枚举出来，用肉眼看清它们的区别。第二个主题是"反例"：当一个看起来很有道理的算法其实是错的，你要能构造一个小输入，当场证明它错。

核心是两兄弟概念：**子数组**（subarray）和**子序列**（subsequence）。举一个具体到数字的例子，输入 `a=[1,2,1]`，先把每个位置标清楚：

| 下标 | 0 | 1 | 2 |
|---|---|---|---|
| 值 | 1 | 2 | 1 |

- 取值 `[1,1]`：作为**子序列**它是合法的——取下标 0 和下标 2，下标是递增的（0 < 2），虽然两个值相同，但它们是两个不同的位置。所以 `[1,1]` 是 `[1,2,1]` 的子序列。
- 同样的 `[1,1]`：作为**子数组**它不合法——子数组必须是原数组里连续的一段，而 `1,1` 这两个值在原数组中并不相邻（中间隔着 2）。

同一个值序列，一个身份合法、另一个身份非法，这就是本章要你刻进脑子的区别。“自动测试”部分 的测试正是用这两条断言钉死这个区别的。

## 2. 基础知识：先读懂这些词

- **子数组（subarray）**：原数组里"连续的一段"。它由起点下标和终点下标唯一确定，中间不许跳过任何元素。
- **子序列（subsequence）**：从原数组里按从左到右的顺序挑若干个元素（可以跳着挑，但不能调换顺序），只要挑出的下标严格递增就行。
- **输入/输出契约**：做题前先白纸黑字写清楚"输入长什么样、输出到底是什么（一个数？一段区间？一个计数？）"。契约不清，后面全白搭。
- **计数与最优化**：计数问题是"这样的东西一共有多少个"，最优化问题是"这些候选里最好的是哪一个"。两者枚举的候选集相同，只是最后一步分别做"数数"还是"挑最大"。
- **必要/充分条件**：满足某条件是"答案正确"所必需的（不满足就一定错），但未必充分（满足了也不一定对）。比如"子数组和最大"必然包含至少一个正数（当数组有正数时），但"含正数"并不足以让一个方案成为最优。
- **贪心反例**：贪心算法每步都做"当前看起来最好"的选择，有时整体却是错的。能证明它错的具体小输入就叫反例。本章练习 2 要求你造一个这样的输入。
- **位掩码（mask）**：用一个整数的每一个二进制位表示"第 i 个元素选不选"。比如 `mask=0b101` 表示选第 0 个和第 2 个元素。它是枚举所有子序列的标准工具。

## 3. 思路推导：从小例子开始

- **Step 1：先确定对象，再确定枚举空间。** 动手写代码前先问自己：题目要的是连续段还是任意挑选？前者枚举空间是"所有下标对 (left, right)"，后者是"每个下标选或不选的全部组合"。
- **Step 2：枚举所有子数组。** 用两层循环枚举端点：`left` 从 0 到 n-1，`right` 从 left+1 到 n（切片是左闭右开），每个非空连续段恰好产出一次。
- **Step 3：数一下子数组有几个。** 长度为 n 的数组，长度为 1 的段有 n 个、长度为 2 的有 n-1 个……长度为 n 的有 1 个，总共 n+(n-1)+...+1 = n(n+1)/2 个。n=3 时就是 3+2+1=6，和下面手算的六段对得上。
- **Step 4：枚举所有子序列。** 把"每个位置选/不选"编码成一个 n 位二进制数 mask，让 mask 从 0 跑到 2ⁿ-1，就一网打尽全部 2ⁿ 种选法（包含全不选的空序列）。
- **Step 5：保留重复值的不同身份。** 枚举时我们按下标决定取舍，而不是按值去重，所以 `[1,2,1]` 里的两个 1 是两个不同的候选来源，`[1,1]` 才会被正确地枚举为子序列。
- **Step 6：用数量公式做交叉验证。** 子数组个数公式 n(n+1)/2 和子序列个数公式 2ⁿ 是我们独立手推出来的，把它们写成断言，就是给枚举器配的"标准答案"（第六节测试正是这么干的）。

**手算演示**（对应 “运行示例”部分 的表格，`a=[1,2,1]`）：

子数组（按 Step 2 的顺序，两层循环 left、right 逐对产出）：

| 轮次 | left | right 边界 | 切片 `nums[left:right]` |
|---|---|---|---|
| 1 | 0 | 1 | `[1]` |
| 2 | 0 | 2 | `[1,2]` |
| 3 | 0 | 3 | `[1,2,1]` |
| 4 | 1 | 2 | `[2]` |
| 5 | 1 | 3 | `[2,1]` |
| 6 | 2 | 3 | `[1]` |

共 3×4/2=6 个，与公式 n(n+1)/2 吻合。

子序列（按 mask 从 0 到 7，二进制位从右往左对应下标 0、1、2）：

| mask（二进制） | 选中的下标 | 结果序列 |
|---|---|---|
| 000 | 无 | `[]`（空） |
| 001 | 0 | `[1]` |
| 010 | 1 | `[2]` |
| 011 | 0,1 | `[1,2]` |
| 100 | 2 | `[1]`（注意与 mask=001 的 `[1]` 是两个不同对象） |
| 101 | 0,2 | `[1,1]` ← 只可能出现在子序列这边 |
| 110 | 1,2 | `[2,1]` |
| 111 | 0,1,2 | `[1,2,1]` |

共 2³=8 个（含空序列）。对照两张表你就亲眼看到：`[1,1]` 在子序列表里、不在子数组表里。

再注意 mask 表里一个小细节：mask=001 产出 `[1]`、mask=100 也产出 `[1]`，两个结果的值一模一样，但它们来自不同的下标组合（{0} 与 {2}），在结果列表里是两个独立的项。这就是"计数按索引、不按值去重"的直观样子。

## 4. 核心代码：enumerate_subarrays

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def enumerate_subarrays(nums):
    return [nums[left:right] for left in range(len(nums)) for right in range(left + 1, len(nums) + 1)]
```

### 逐段读懂代码

### 函数 1：`enumerate_subarrays`（枚举全部非空子数组）

```python
def enumerate_subarrays(nums):
    return [nums[left:right] for left in range(len(nums)) for right in range(left + 1, len(nums) + 1)]
```

- 这是一行列表推导式，展开后就是普通的两层循环：外层枚举起点 `left`（0 到 n-1），内层枚举终点边界 `right`（left+1 到 n）。展开成等价的普通写法长这样，对照着看更直白：

```python
result = []

for left in range(len(nums)):
    for right in range(left + 1, len(nums) + 1):
        result.append(nums[left:right])

return result
```

- 切片 `nums[left:right]` 是左闭右开的：含下标 left、不含下标 right。所以 `right` 的上限要写成 `len(nums)+1` 才能让切片真正取到最后一个元素；同理 `right` 从 `left+1` 起，保证切片至少含一个元素（空段被自动排除）。
- 返回值是"子数组的列表"，每个子数组是一次真实的切片拷贝，重复值各自成项，没有按值去重。

### 函数 2：`enumerate_subsequences`（枚举全部子序列，含空）

```python
def enumerate_subsequences(nums):
    result = []
    for mask in range(1 << len(nums)):
        result.append([x for i, x in enumerate(nums) if mask >> i & 1])
    return result
```

- `result=[]`：先准备一个空列表装结果。
- `for mask in range(1<<len(nums))`：`1<<n` 是位运算写法，意思是 2ⁿ（把 1 左移 n 位）。mask 从 0 跑到 2ⁿ-1，恰好覆盖 n 个二进制位的全部取值，也就是"每个元素选/不选"的全部方案。以 n=3 为例，mask 依次取 0,1,2,...,7，正好对应第三节表格里的 000 到 111 八行。
- `[x for i,x in enumerate(nums) if mask>>i & 1]`：`enumerate` 同时给出下标 i 和值 x；`mask>>i & 1` 先把 mask 右移 i 位、再与 1 按位与，取出 mask 第 i 位上的二进制数字——是 1 就代表"选中下标 i"。整个推导式的意思是"留下所有被选中的元素，按原顺序组成列表"。拿 mask=5（二进制 101）走一遍：第 0 位是 1，选中下标 0（值 1）；第 1 位是 0，跳过下标 1；第 2 位是 1，选中下标 2（值 1）；结果 `[1,1]`。
- `result.append(...)` 把每种选法存进结果；循环走完返回全部 2ⁿ 个子序列（mask=0 时产出空列表 `[]`，这是有意保留的——含空序列，调用方需要时自己过滤即可）。

顺带一提，“运行示例”部分 里 `show_table` 把两个枚举器在 `a=[1,2,1]` 上的产出画成了对比表（6 个非空子数组 vs 8 个含空子序列），你可以对照第三节的表格逐行核对。

## 5. 分段实现：按顺序运行

**本章接口：** `enumerate_subarrays(nums)`、`enumerate_subsequences(nums)`

### enumerate_subarrays

In [1]:
def enumerate_subarrays(nums):
    return [nums[left:right] for left in range(len(nums)) for right in range(left + 1, len(nums) + 1)]

### enumerate_subsequences

In [2]:
def enumerate_subsequences(nums):
    result = []
    for mask in range(1 << len(nums)):
        result.append([x for i, x in enumerate(nums) if mask >> i & 1])
    return result

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,2,1]
show_table(['对象','数量','候选值序列'], [('非空子数组',len(enumerate_subarrays(a)),enumerate_subarrays(a)),('含空子序列',len(enumerate_subsequences(a)),enumerate_subsequences(a))])

对象,数量,候选值序列
非空子数组,6,"[[1], [1, 2], [1, 2, 1], [2], [2, 1], [1]]"
含空子序列,8,"[[], [1], [2], [1, 2], [1], [1, 1], [2, 1], [1, 2, 1]]"


## 7. 正确性、适用条件与复杂度

子数组由两个端点确定；子序列由每个索引的二元选择确定。输出中保留重复值列表是有意的，因为计数按索引而不是按值去重。

**代价：** 显式复制全部子数组总 O(n³) 元素输出；全部子序列 O(n2ⁿ) 时间和输出空间。

### 展开理解

**为什么是对的：** 先看子数组。一个连续段完全由"从哪开始、到哪结束"两个下标决定，我们的两层循环把每一对满足"起点在终点前面"的下标都列了一遍，而且每对只列一遍，所以产出的恰好是全部非空子数组，没有遗漏也没有重复。再看子序列。一个子序列完全由"每个位置选还是不选"这一串二选一决定，mask 从 0 到 2ⁿ-1 恰好把每一种选与不选的组合都轮过了一遍，一一对应到每一个子序列。还有一点容易忽略：我们坚持按下标取舍而不是按值取舍，所以 `[1,2,1]` 的两个 1 虽然值相同，身份不同，各自参与组合——这正是不该按值去重的原因，计数时算的是"下标组合数"而不是"不同值序列数"。

**复杂度是多少：** 子数组共有 n(n+1)/2 个，平均每个长度约 n/3，把它们全部真实拷贝出来，总共要搬运约 O(n³) 个元素。子序列共有 2ⁿ 个，产出它们至少要写 2ⁿ 个列表，每个平均又含 n/2 个元素，所以时间和输出空间都是 O(n·2ⁿ)。举个具体的数：n=20 时子序列有 2²⁰≈一百万个，程序还能跑；n=30 时就有约十亿个，基本跑不动了。这就是"枚举只是理解工具、真正解题要靠更聪明的算法"的定量原因。

直观感受一下两类对象的规模差距：

| n | 子数组个数 n(n+1)/2 | 子序列个数 2ⁿ |
|---|---|---|
| 3 | 6 | 8 |
| 10 | 55 | 1024 |
| 20 | 210 | 约 105 万 |

n 才到 20，两者就差了五千倍——子序列的指数增长远比子数组的平方增长凶猛。

**一个高频误区——"按值去重"错在哪：** 有人觉得 `[1]` 出现两次（mask=001 与 mask=100）是"重复"，想去掉一份。但去重后计数就从 2ⁿ 变成了"不同值序列数"，公式对不上、测试立刻失败；更重要的是语义变了——比如"统计有多少种方式选出和为 k 的子序列"这类计数题，两种选法是两个不同的答案。所以本章明确约定：对象按下标区分身份，值相同也是两个不同的候选。

## 8. 单元测试：每个用例在检查什么

```python
for n in range(7):
    a = [1] * n
    assert len(enumerate_subarrays(a)) == n * (n + 1) // 2
    assert len(enumerate_subsequences(a)) == 1 << n

assert [1, 1] in enumerate_subsequences([1, 2, 1])

assert [1, 1] not in enumerate_subarrays([1, 2, 1])

print('N016: 所有本章断言通过')
```

- 第一段 for 循环是**数量公式核对**：对 n=0..6，各造一个长度为 n 的全 1 数组（全 1 是故意的——元素全相同，任何按值去重的错误实现都会把数量算少，从而暴露出来）。它断言子数组个数等于 n(n+1)//2（等差数列求和，`//` 是整除），子序列个数等于 1<<n（即 2ⁿ）。这两个公式是我们在 Step 3、Step 4 手推出来的，测试就是拿公式当标准答案。顺带留意 n=0 那一轮：空数组的子数组个数是 0、子序列个数是 1（就是那个空序列），公式也恰好给出 0×1/2=0 和 2⁰=1，边界自然吻合。
- `assert [1,1] in enumerate_subsequences([1,2,1])`：这是**特殊值测试**，验证 `[1,1]` 必须作为子序列出现（下标 0 和 2 组合）。
- `assert [1,1] not in enumerate_subarrays([1,2,1])`：与上一条正相对，验证 `[1,1]` 绝不能作为连续段出现。两条合起来，把"子序列 ≠ 子数组"这个本章核心结论钉进了测试。注意 `in` 和 `not in` 比较的是列表相等：只要枚举结果里存在（或不存在）值为 `[1,1]` 的项，断言就成立。
- `print(...)` 打印一句通过确认，方便肉眼检查。

In [4]:
for n in range(7):
    a = [1] * n
    assert len(enumerate_subarrays(a)) == n * (n + 1) // 2
    assert len(enumerate_subsequences(a)) == 1 << n

assert [1, 1] in enumerate_subsequences([1, 2, 1])

assert [1, 1] not in enumerate_subarrays([1, 2, 1])

print('N016: 所有本章断言通过')

N016: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把最大子数组改成最大子序列并解释变化。

**练习 2：** 反驳遇到正数就选的错误连续算法。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（把最大子数组改成最大子序列）**：思路方向——最大子序列和其实有一个极简单的结论：把所有正数都选上就是最优（如果全是非正数就只选最大的那一个）。提示：先用小输入手算验证这个直觉，比如 `[1,-2,3]` 的最大子序列和是 1+3=4，而最大子数组和是 3；再对比两个答案为什么会不同（子序列可以跳过中间的 -2，子数组不能）。别忘了写清边界：空序列允不允许？全负数组怎么办？
- **练习 2（反驳"遇到正数就选"的错误连续算法）**：思路方向——设想一个错误算法："扫描数组，见正数就纳入当前段，遇到负数就丢掉当前段重新开始"。提示：构造一个中间有"负数但后面有更大回报"的输入，比如 `[3,-1,4]`——正确答案是整段 3-1+4=6，而这个见负就丢的算法只会得到 4 或 3。你先用 `enumerate_subarrays` 穷举全部候选算出真最优，再展示错误算法在这个输入上的输出，两者一对比，反例就成立了。

写反例时的验收标准：反例要小到可以一眼验算（三个元素足够），并且要说清"错误算法的输出"和"穷举出的真最优"分别是多少、差在哪里。反例一摆出来，错误算法就再也没法辩解了。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def enumerate_subarrays(nums):
    return [nums[left:right] for left in range(len(nums)) for right in range(left + 1, len(nums) + 1)]

def enumerate_subsequences(nums):
    result = []
    for mask in range(1 << len(nums)):
        result.append([x for i, x in enumerate(nums) if mask >> i & 1])
    return result

# 示例与回归测试
for n in range(7):
    a = [1] * n
    assert len(enumerate_subarrays(a)) == n * (n + 1) // 2
    assert len(enumerate_subsequences(a)) == 1 << n

assert [1, 1] in enumerate_subsequences([1, 2, 1])

assert [1, 1] not in enumerate_subarrays([1, 2, 1])

print('N016: 所有本章断言通过')

N016: 所有本章断言通过


## 11. 代表题与迁移

- **53. Maximum Subarray（最大子数组）**：这题练的是"连续段"这个建模对象——你必须枚举/优化的是子数组，本章的子数组枚举器就是它的暴力基线。
- **300. Longest Increasing Subsequence（最长递增子序列）**：这题练的是"子序列"这个建模对象——挑元素可以跳着挑但不许换顺序，正好用本章"按下标选/不选"的视角去理解（本章只做预览，不要求现在就能解）。
- 迁移建议：以后拿到任何题，先用 30 秒回答"题目要的对象是子数组、子序列还是别的"，再决定枚举空间。这一个习惯能提前掐灭一大类"算法没错、建模错了"的 bug。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。